# Hückel level

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print('Ready')

---
## Hückel MOs and energies

For any conjugated molecule, the Hückel hamiltonian reads :

$$H = \alpha\,I + \beta\,T$$

where $T$ is the **topological matrix**.
Eigenvalues $x_k$ of $T$ give the energies $E_k = \alpha + x_k\,\beta$, and eigenvectors the LCAO coefficients.

| Function | Description |
|---|---|
| `topo_cycle(N)` / `topo_chain(N)` | Topological matrix of a cycle or a chain |
| `huckel_from_topo(T, n_e)` | Solves H = αI + βT → energies, OM, occupations, E_π |
| `plot_energy_levels(ax, ev, occ)` | Aufbau diagram |
| `plot_mo_coefficients(ax, k, …)` | LCAO of a MO |
| `plot_huckel_topo(T, n_e, …)` | Full figure |

---

In [ ]:
def topo_cycle(N):
    """Adjacency matrix of a cycle with N atoms."""
    T = np.zeros((N, N))
    for i in range(N):
        T[i, (i+1) % N] = T[(i+1) % N, i] = 1
    return T

def topo_chain(N):
    """Adjacency matrix of a linear chain with N atoms."""
    T = np.zeros((N, N))
    for i in range(N - 1):
        T[i, i+1] = T[i+1, i] = 1
    return T


def huckel_from_topo(T, n_electrons, alpha=0.0, beta=-1.0):
    """Solves H = alpha*I + beta*T.

    Returns (energies, MOs, occupations, E_pi_total) with MOs[:, k]
    the LCAO coefficients of the k-th MO (sorted by increasing energy).
    The sign is fixed so that the largest coefficient (in absolute value) is > 0.
    """
    N       = T.shape[0]
    ev, vec = np.linalg.eigh(alpha * np.eye(N) + beta * T.astype(float))
    order   = np.argsort(ev)
    ev, vec = ev[order], vec[:, order]
    for k in range(N):
        if vec[np.argmax(np.abs(vec[:, k])), k] < 0:
            vec[:, k] *= -1
    occ = np.zeros(N)
    rem = int(n_electrons)
    for k in range(N):
        if rem <= 0: break
        ne = min(2, rem); occ[k] = ne; rem -= ne
    return ev, vec, occ, float(np.dot(occ, ev))


def _deg_groups(energies, tol=1e-5):
    """Groups the indices of degenerate energy levels."""
    used, groups = [False] * len(energies), []
    for i in range(len(energies)):
        if used[i]: continue
        g = [i]
        for j in range(i + 1, len(energies)):
            if not used[j] and abs(energies[j] - energies[i]) < tol:
                g.append(j); used[j] = True
        used[i] = True; groups.append(g)
    return groups


def plot_energy_levels(ax, energies, occupations, alpha=0.0, beta=-1.0, title=None):
    """Diagram of energy levels (y-axis) with electron arrows Aufbau.

    Colors: HOMO = green, LUMO = orange, occupied = blue, virtual = gray.
    """
    N      = len(energies)
    groups = _deg_groups(energies)
    occ_k  = [k for k in range(N) if occupations[k] > 0]
    virt_k = [k for k in range(N) if occupations[k] == 0]
    homo   = max(occ_k) if occ_k else None
    lumo   = min(virt_k) if virt_k else None

    e_span = (max(energies) - min(energies)) or 1.0
    pad    = e_span * 0.20
    ah     = e_span * 0.09

    ax.set_xlim(-0.9, 1.2)
    ax.set_ylim(min(energies) - pad, max(energies) + pad)
    ax.set_ylabel('Energy  (α + x·β)', fontsize=8)
    ax.set_xticks([])
    for sp in ('top', 'right', 'bottom'):
        ax.spines[sp].set_visible(False)
    if title:
        ax.set_title(title, fontsize=9)

    for group in groups:
        ndeg = len(group)
        xs   = np.linspace(-0.25, 0.25, ndeg) if ndeg > 1 else [0.0]
        for k, xc in zip(group, xs):
            e  = energies[k]
            ne = int(occupations[k])
            lc = ('forestgreen' if k == homo else
                  'darkorange'  if k == lumo else
                  'steelblue'   if ne > 0    else 'gray')
            ax.hlines(e, xc - 0.22, xc + 0.22, colors=lc, linewidth=2.5, zorder=3)
            if ne >= 1:
                ax.annotate('', xy=(xc - 0.07, e + ah),
                            xytext=(xc - 0.07, e - ah * 0.3),
                            arrowprops=dict(arrowstyle='->', color='black',
                                           lw=1.1, mutation_scale=9), zorder=4)
            if ne >= 2:
                ax.annotate('', xy=(xc + 0.07, e - ah),
                            xytext=(xc + 0.07, e + ah * 0.3),
                            arrowprops=dict(arrowstyle='->', color='black',
                                           lw=1.1, mutation_scale=9), zorder=4)
            x_k = (e - alpha) / beta
            lbl = f' x = {x_k:+.4f}'
            if k == homo:   lbl += '  HOMO'
            elif k == lumo: lbl += '  LUMO'
            ax.text(xc + 0.25, e, lbl, va='center', fontsize=6.5, color=lc)


def plot_mo_coefficients(ax, k, energy, coeffs, occupation,
                         atom_labels=None, alpha=0.0, beta=-1.0):
    """Bar plots of the LCAO coefficients of a MO (blue ≥ 0, red < 0)."""
    N   = len(coeffs)
    lbs = atom_labels or [str(i + 1) for i in range(N)]
    ax.bar(np.arange(N), coeffs,
           color=['steelblue' if c >= 0 else 'tomato' for c in coeffs],
           edgecolor='black', linewidth=0.4, width=0.7)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(np.arange(N))
    ax.set_xticklabels(lbs, fontsize=6.5)
    ax.tick_params(axis='y', labelsize=6)
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    x_k    = (energy - alpha) / beta
    ne_str = '↑↓' if int(occupation) == 2 else ('↑' if int(occupation) == 1 else '∅')
    ax.set_title(f'ψ{k+1}   x = {x_k:+.4f}   {ne_str}', fontsize=7.5)


def plot_huckel_topo(T, n_electrons, atom_labels=None, alpha=0.0, beta=-1.0,
                     mol_name='Molecule'):
    """Complete figure: energy level diagram + LCAO coefficients of all MOs.

    Returns (fig, energies, MOs, occupations, E_pi_total).
    """
    ev, MOs, occ, E_tot = huckel_from_topo(T, n_electrons, alpha, beta)
    N      = T.shape[0]
    labels = atom_labels or [f'C{i+1}' for i in range(N)]

    n_cols = min(N, 4)
    n_rows = (N + n_cols - 1) // n_cols
    fig    = plt.figure(figsize=(3.2 + 2.8 * n_cols, max(5, 2.2 * n_rows + 1.0)))
    gs     = fig.add_gridspec(n_rows, n_cols + 1,
                              hspace=0.55, wspace=0.42,
                              width_ratios=[1.4] + [1] * n_cols)
    plot_energy_levels(fig.add_subplot(gs[:, 0]), ev, occ, alpha, beta,
                       title="Energy levels")
    for k in range(N):
        r, c = divmod(k, n_cols)
        plot_mo_coefficients(fig.add_subplot(gs[r, c + 1]),
                             k, ev[k], MOs[:, k], occ[k], labels, alpha, beta)
    for k in range(N, n_rows * n_cols):
        r, c = divmod(k, n_cols)
        fig.add_subplot(gs[r, c + 1]).axis('off')

    fig.suptitle(f'{mol_name}  —  {n_electrons} electrons π   '
                 f'E_π = {E_tot:.4f}  (α = {alpha}, β = {beta})',
                 fontsize=10, y=1.02)
    plt.tight_layout()
    return fig, ev, MOs, occ, E_tot

### Examples

Dafine the **topological matrix** $T$ and call `plot_huckel_topo`.

Helpers : `topo_cycle(N)` and `topo_chain(N)`

For any molecule, you can build $T$ manually :
```python
T = np.zeros((N, N))
for i, j in bond_list:
    T[i, j] = T[j, i] = 1
```

In [ ]:
topological_matrix=np.array([[0, 1],
                                [1, 0]])
labels=['C1','C2']
name='Ethylene'

fig, ev, MOs, occ, E_tot = plot_huckel_topo(topological_matrix, 2, atom_labels=labels, mol_name=name)
plt.show()

# Add hexatriene and compare to benzene

---
## Hückel Hamiltonian with flux

In the standard Hückel model for an $N$-atom ring, the resonance integrals are $\beta < 0$. In the presence of a reduced magnetic flux $\Phi = \Phi_{mag}/\Phi_0$, each hopping term becomes:

$$\beta_{i,i+1} \longrightarrow \beta\, e^{i\,2\pi\Phi/N}$$

The matrix $H$ remains **Hermitian** ($H_{ij} = H_{ji}^*$) and its eigenvalues remain real, but the eigenfunctions become complex.

The bond current between atoms $i$ and $j=i+1$ is:

$$J_{i \to j} = \frac{2e}{\hbar}\sum_{k\,\text{occ}} n_k\,\text{Im}\left(c_{ki}^*\, \beta\, e^{i2\pi\Phi/N}\, c_{kj}\right)$$

where $n_k \in \{1, 2\}$ is the number of electrons in orbital $k$ and $c_{ki}$ the LCAO coefficients.

In [ ]:
def huckel_hamiltonian(N, Phi, alpha=0.0, beta=-1.0):
    """Hückel Hamiltonian with Peierls phase for an N-atom ring."""
    phase = # fill in the pure imagnary in python is just 1j
    H     = np.zeros((N, N), dtype=complex)
    np.fill_diagonal(H, alpha)
    for i in range(N):
        j       = (i + 1) % N
        H[i, j] = # fill in
        H[j, i] = # fill in
    return H


def solve_huckel(N, n_electrons, Phi, beta=-1.0):
    """Solve Hückel + Peierls and return orbitals, occupations, total energy."""
    H       = huckel_hamiltonian(N, Phi, beta=beta)
    ev, vec = np.linalg.eigh(H)
    idx     = np.argsort(ev)
    ev      = ev[idx]
    vec     = vec[:, idx]

    occ_vecs, occ_ne = [], []
    E_tot, remaining = 0.0, n_electrons
    for k in range(N):
        if remaining <= 0: break
        ne = min(2, remaining)
        E_tot += ne * ev[k]
        occ_vecs.append(vec[:, k])
        occ_ne.append(ne)
        remaining -= ne
    return ev, occ_vecs, occ_ne, E_tot


def bond_currents(N, occ_vecs, occ_ne, Phi, beta=-1.0):
    """Current on each bond i -> i+1 (units e/hbar)."""
    phase  = np.exp(1j * 2 * np.pi * Phi / N)
    J      = np.zeros(N)
    for psi, ne in zip(occ_vecs, occ_ne):
        for i in range(N):
            j      = (i + 1) % N
            J[i]  += ne * 2 * np.imag(np.conj(psi[i]) * beta * phase * psi[j])
    return J


---
## 2 — Visualisation function: arrows on bonds

In [ ]:
def atom_positions(N, R=1.0, offset=np.pi/2):
    """Coordinates of the N atoms on a circle of radius R."""
    angles = np.array([2*np.pi*k/N + offset for k in range(N)])
    return R*np.cos(angles), R*np.sin(angles)


def draw_molecule_current(ax, N, n_electrons, Phi, atom_labels=None,
                          beta=-1.0, R=1.0, title=None):
    """
    Draw the N-atom ring with bond currents represented as
    coloured arrows proportional to the current magnitude.
    """
    ev, vecs, ne_l, E_tot = solve_huckel(N, n_electrons, Phi, beta)
    J = bond_currents(N, vecs, ne_l, Phi, beta)
    J_max = np.max(np.abs(J)) if np.max(np.abs(J)) > 1e-10 else 1.0

    xs, ys = atom_positions(N, R)

    ax.set_aspect('equal')
    ax.axis('off')
    lim = R * 1.55
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)

    # Bonds + arrows
    for i in range(N):
        j  = (i + 1) % N
        x0, y0 = xs[i], ys[i]
        x1, y1 = xs[j], ys[j]

        # Bond line
        ax.plot([x0, x1], [y0, y1], 'k-', linewidth=1.5, zorder=1)

        # Arrow thickness and colour
        lw_arrow = 1.5 + 5.0 * abs(J[i]) / J_max
        color    = 'steelblue' if J[i] < 0 else 'tomato'

        # Arrow start and end (mid-bond, offset from atoms)
        frac  = 0.15   # offset from each atom
        sx    = x0 + frac*(x1-x0)
        sy    = y0 + frac*(y1-y0)
        ex    = x1 - frac*(x1-x0)
        ey    = y1 - frac*(y1-y0)

        # Arrow direction based on current sign
        # J[i] < 0: current flows from i to j (clockwise in standard layout)
        if J[i] < 0:
            ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                        arrowprops=dict(arrowstyle='->', color=color,
                                       lw=lw_arrow, mutation_scale=15),
                        zorder=4)
        elif J[i] > 0:
            ax.annotate('', xy=(sx, sy), xytext=(ex, ey),
                        arrowprops=dict(arrowstyle='->', color=color,
                                       lw=lw_arrow, mutation_scale=15),
                        zorder=4)
        # Numerical value on the bond
        mx, my = (x0+x1)/2, (y0+y1)/2
        # Perpendicular outward offset
        nx = -(y1-y0); ny = (x1-x0)
        norm_n = np.sqrt(nx**2+ny**2)
        ax.text(mx + 0.18*nx/norm_n, my + 0.18*ny/norm_n,
                f'{J[i]:+.3f}', ha='center', va='center',
                fontsize=6.5, color=color)

    # Atoms
    labels = atom_labels if atom_labels else ['C']*N
    for k, (x, y, lbl) in enumerate(zip(xs, ys, labels)):
        color_atom = 'lightyellow' if lbl == 'C' else \
                     ('#aad4f5'    if lbl == 'N' else '#f5d5aa')
        ax.add_patch(plt.Circle((x, y), 0.13, color=color_atom, # type: ignore
                                ec='black', lw=1.5, zorder=5))
        ax.text(x, y, lbl, ha='center', va='center',
                fontsize=8, fontweight='bold', zorder=6)

    # Title and legend
    J_mean = np.mean(J)
    sens   = 'diamagnetic' if J_mean < -1e-4 else \
             ('paramagnetic' if J_mean > 1e-4 else 'zero')
    col_s  = 'steelblue' if J_mean < 0 else 'tomato'
    if title:
        ax.set_title(title, fontsize=9, pad=4)
    ax.text(0, -lim + 0.08,
            f'J = {J_mean:+.4f}  ({sens})',
            ha='center', fontsize=8.5, color=col_s, fontweight='bold')

    return J

---
## 3 — Benzene vs cyclobutadiene comparison

In [ ]:
Phi = 0.05
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

draw_molecule_current(axes[0], N=6, n_electrons=6, Phi=Phi,
                      title=f'Benzene (6 e\u207b, 4k+2)\n\u03a6 = {Phi}')
draw_molecule_current(axes[1], N=4, n_electrons=4, Phi=Phi,
                      title=f'Cyclobutadiene (4 e\u207b, 4k)\n\u03a6 = {Phi}')

# Shared legend
dia  = mpatches.Patch(color='steelblue', label='Diamagnetic current (\u21bb clockwise, aromatic)')
para = mpatches.Patch(color='tomato',    label='Paramagnetic current (\u21ba anticlockwise, antiaromatic)')
fig.legend(handles=[dia, para], loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Hückel–Peierls bond currents\n'
             'Arrow thickness \u221d current magnitude', fontsize=11)
plt.tight_layout()
plt.show()